# EDA del CSI con pandas
De datos crudos I/Q a CSI, visualización de subportadoras, espectro y preprocesamiento.

In [ ]:
import os, glob, re, ast
import numpy as np, pandas as pd, scipy.io as scio
import matplotlib.pyplot as plt

HERE = os.path.abspath('')
DATA = os.path.join(HERE, '..', 'mmfi_dataset')
CRUDAS = os.path.join(HERE, '..', 'capturas_crudas')
ACT = {'A01':'habitacion','A02':'estatica','A03':'sentadilla','A04':'caminar','A05':'torso'}

## 1. Datos crudos I/Q → CSI (un paquete)

In [ ]:
f = glob.glob(os.path.join(CRUDAS, 'sesion2_estatica', 'csi', 'rx0.csv'))[0]
for line in open(f, encoding='utf-8', errors='ignore'):
    if 'CSI_DATA' in line:
        arr = ast.literal_eval(line[line.index('['):line.rindex(']')+1])
        if len(arr) == 384: break
a = np.array(arr, float)
comp = a[1::2] + 1j*a[0::2]                                 # 192 complejos (I=impar, Q=par)
csi = np.concatenate([comp[66:123], comp[134:191]])        # 114 subportadoras HT40
df_iq = pd.DataFrame({'subportadora': range(114), 'I': csi.real, 'Q': csi.imag,
                      'amplitud': np.abs(csi), 'fase': np.angle(csi)})
df_iq.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
df_iq.plot(x='subportadora', y='amplitud', ax=ax[0], legend=False, color='#1C7293', title='Amplitud')
df_iq.plot(x='subportadora', y='fase', ax=ax[1], legend=False, color='#B85042', title='Fase')
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 2. Dataset → DataFrame y estadísticas

In [ ]:
filas = []
for mat in sorted(glob.glob(os.path.join(DATA, '**', 'wifi-csi', '*.mat'), recursive=True)):
    acc = os.path.basename(os.path.dirname(os.path.dirname(mat)))
    d = scio.loadmat(mat); amp = d['CSIamp'].astype(float); pha = d['CSIphase'].astype(float)
    amp[np.isinf(amp)] = np.nan; pha[np.isinf(pha)] = np.nan
    am = np.nanmean(amp, axis=2); pm = np.nanmean(pha, axis=2)
    for rx in range(3):
        filas.append({'actividad': ACT.get(acc, acc), 'rx': f'rx{rx}',
                      'amp_media': np.nanmean(am[rx]), 'amp_std': np.nanstd(am[rx]), 'fase_media': np.nanmean(pm[rx])})
df = pd.DataFrame(filas)
df.groupby(['actividad','rx'])[['amp_media','amp_std','fase_media']].mean().round(3)

## 3. CSI por subportadora y espectro

In [ ]:
perfiles = {}
for acc, nom in ACT.items():
    mats = sorted(glob.glob(os.path.join(DATA,'E01','S01',acc,'wifi-csi','*.mat')))[:300]
    if mats: perfiles[nom] = np.nanmean([np.nanmean(scio.loadmat(m)['CSIamp'][0].astype(float),axis=1) for m in mats], axis=0)
pd.DataFrame(perfiles).plot(figsize=(12,4), title='Amplitud media por subportadora (rx0)')
plt.xlabel('subportadora'); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
M = np.array([np.nanmean(scio.loadmat(m)['CSIamp'][0].astype(float),axis=1)
              for m in sorted(glob.glob(os.path.join(DATA,'E01','S01','A03','wifi-csi','*.mat')))[:250]])
plt.figure(figsize=(12,4)); plt.imshow(M.T, aspect='auto', origin='lower', cmap='viridis')
plt.title('Espectro CSI (subportadora x tiempo) - sentadilla'); plt.xlabel('frame'); plt.ylabel('subportadora')
plt.colorbar(); plt.tight_layout(); plt.show()

## 4. Preprocesamiento (normalización)

In [ ]:
An = np.array([(r-np.nanmin(r))/(np.nanmax(r)-np.nanmin(r)+1e-9) for r in M])
As = np.array([(r-np.nanmean(r))/(np.nanstd(r)+1e-9) for r in An])
fig, ax = plt.subplots(1,3, figsize=(16,4))
for a,Mx,t in [(ax[0],M,'cruda'),(ax[1],An,'normalizado [0,1]'),(ax[2],As,'estandarizado')]:
    im=a.imshow(Mx.T, aspect='auto', origin='lower', cmap='viridis'); a.set_title(t); a.set_xlabel('frame')
plt.tight_layout(); plt.show()